In [21]:
import cvxpy as cp
import numpy as np
import time

In [ ]:
def read_data(filepath):
    data = dict()
    data["F"] = []
    data["P"] = []

    with open(filepath,'r') as infile:
        infile.readline()
        data["n"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["v0"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["vmin"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["vmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["tmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["dmax"] = int(infile.readline())
        infile.readline()
        
        infile.readline()
        data["mmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["et"] = float(infile.readline())
        infile.readline()

        infile.readline()
        data["me"] = float(infile.readline())
        infile.readline()

        infile.readline()
        data["tdmin"]= int(infile.readline())
        infile.readline()

        infile.readline()
        data["vtmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["vdmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        for _ in range(data["n"]):
            data["F"].append(float(infile.readline()))
        infile.readline()

        infile.readline()
        for _ in range(data["n"]):
            data["P"].append(float(infile.readline()))
        
        data["F"] = np.array(data["F"])
        data["P"] = np.array(data["P"])
        
    return data

In [9]:
read_data("BelgiumScenario1_15jours.txt")

{'F': array([ 93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         93975.77028029,  93975.77028029,  93975.77028029,  93975.77028029,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         92945.33447753,  92945.33447753,  92945.33447753,  92945.33447753,
         96654.30561556,  96654.30561556,  96654.30561556,  96654.30561556,
       

In [46]:
def hydro_with_cost(data, C_jour = 50000):
    params = read_data(data)
    N = int(params['n']) 
    
    
    #Decision variables
    T = cp.Variable(N, nonneg=True, name="Turbinage")
    M = cp.Variable(N, nonneg=True, name="Pompage")
    D = cp.Variable(N, nonneg=True, name="Délestage")
    V = cp.Variable(N, nonneg=True, name="Volume")

    #added binary decision variable
    c = cp.Variable(15, boolean=True, name="Activation de la Turbine (jour)")
    
    #Objective function
    profits = params['P'] @ (params['et'] * T - params['me'] * M)
    #with cost
    if C_jour != 0:
        profits = params['P'] @ (params['et'] * T - params['me'] * M) - (cp.sum(c * C_jour))

    objective = cp.Maximize(profits)
    
    #list of constraints
    constraints = []

    #Daily turbine cost constraint
    if C_jour != 0:
        for j in range(15):
            for h in range(24):
                t = j * 24 + h
                constraints.append(T[t] <= params['tmax'] * c[j])
    
    for t in range(N):
        
        #Limits 
        constraints += [V[t] >= params['vmin'], V[t] <= params['vmax']]
        constraints += [T[t] <= params['tmax']]
        constraints += [M[t] <= params['mmax']]
        constraints += [D[t] <= params['dmax']]
        
        #TDlimit
        constraints += [T[t] + D[t] >= params['tdmin']]

        if t > 0:
            #volume reservoir (V0 quand t == 0)
            constraints += [V[t] == V[t-1] + params['F'][t] + M[t] - T[t] - D[t]]
            #Derivative limits
            constraints += [(T[t] - T[t-1]) <= params['vtmax']]
            constraints += [(T[t] - T[t-1]) >= -params['vtmax']]
            constraints += [(D[t] - D[t-1]) <= params['vdmax']]
            constraints += [(D[t] - D[t-1]) >= -params['vdmax']]
            
    #solver
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)
    
    sol = {
        "C": c.value,
        "V": V.value,       
        "T": T.value,       
        "D": D.value,       
        "M": M.value,       
        "valopt": prob.value 
    }
    
    return sol

In [31]:
result = hydro_with_cost("BelgiumScenario1_15jours.txt",0)
print(f"Bénéfice optimal : {result['valopt']:.2f} €")
result




Bénéfice optimal : 1885335.89 €


{'V': array([6000000.        , 5843975.77028029, 5887951.54056058,
        5931927.31084087, 5975903.08112116, 6000000.        ,
        5824096.91887884, 5518072.68915913, 5212048.45943942,
        5056024.22971971, 5100000.        , 5143975.77028029,
        5187951.54056058, 5231927.31084087, 5275903.08112116,
        5448193.83775768, 5492169.60803797, 5536145.37831826,
        5380121.14859855, 5074096.91887884, 4768072.68915913,
        4462048.45943942, 4156024.22971971, 4000000.        ,
        4042945.33447753, 4085890.66895506, 4128836.00343258,
        4671781.33791011, 4714726.67238764, 4757672.00686517,
        4600617.34134269, 4293562.67582022, 4046781.33791011,
        4000000.        , 4042945.33447753, 4085890.66895506,
        4128836.00343258, 4671781.33791011, 5214726.67238764,
        5757672.00686517, 6000000.        , 5907692.28853615,
        5615384.5770723 , 5308329.91154983, 5001275.24602736,
        4694220.58050489, 4387165.91498241, 4280111.24945994,
   

In [41]:
num_runs = 10
run_times = []

for run in range(num_runs):
    start_time = time.time()
    hydro_with_cost("BelgiumScenario1_15jours.txt",0)
    end_time = time.time()
    
    run_times.append(end_time-start_time)

average_time = np.mean(run_times)

print(f"Average execution time: {average_time:.4f} seconds")































Average execution time: 2.2875 seconds


In [88]:
import plotly.graph_objects as ui
from plotly.subplots import make_subplots
import kaleido


def plot_hydro_profits(solutions_dict, data):
    params = read_data(data) if isinstance(data, str) else data
    P = np.array(params["P"])
    et = params["et"]
    me = params["me"]

    fig = make_subplots()

    # Define unique colors for each line to tell them apart easily
    colors = {
        0: "#ff595e",  
        50000: "#b99328",  
        150000: "#8ac926",  
    }

    # Track overall maximum hours to set up the axis ranges and day lines
    max_N = 0

    # Iterate through each pre-calculated case passed into the function
    for C_jour, solution in solutions_dict.items():
        T = np.array(solution["T"])
        M = np.array(solution["M"])

        N = len(T)
        max_N = max(max_N, N)
        hours = np.arange(N)

        # Calculate specific profiles for this instance
        hourly_operational_profit = P * (et * T - me * M)
        hourly_profit = hourly_operational_profit.copy()

        # Deduct fixed costs on active days
        for j in range(15):
            day_slice = slice(j * 24, (j + 1) * 24)
            if np.max(T[day_slice]) > 1e-3:
                hourly_profit[day_slice] -= C_jour / 24

        cumulative_profit = np.cumsum(hourly_profit)
        final_profit = cumulative_profit[-1]
        color = colors.get(C_jour, "rgb(127, 127, 127)")

        # Add the line trace for this specific C_jour scenario
        fig.add_trace(
            ui.Scatter(
                x=hours,
                y=cumulative_profit,
                name=f"C_jour = {C_jour:,} €",
                mode="lines",
                line=dict(color=color, width=3),
            ),
        )

        # --- END LABEL FOR EACH LINE ---
        fig.add_annotation(
            x=hours[-1],
            y=final_profit,
            text=f"{final_profit:,.0f} €",
            showarrow=True,
            arrowhead=2,
            arrowcolor=color,
            arrowsize=0.8,
            ax=45,  # Push labels out to the right in the padded area
            ay=0,  # Keep them vertically level with the line tip
            font=dict(size=11, color="white"),
            bgcolor=color,
            bordercolor=color,
            borderwidth=1,
            borderpad=4,
        )

    # --- GENERATE VERTICAL DAY SEGMENT SHAPES ---
    day_lines = []
    for hour in range(24, max_N, 24):
        day_lines.append(
            dict(
                type="line",
                xref="x",
                yref="paper",
                x0=hour,
                y0=0,
                x1=hour,
                y1=1,
                line=dict(
                    color="rgba(100, 100, 100, 0.4)", width=1.5, dash="dash"
                ),
            )
        )

    # --- GLOBAL GRAPH UI TWEAKS ---
    fig.update_layout(
        title="Profit Cumulatif Optimal Selon le Coût Journalier de la Turbine",
        xaxis_title="Temps (heures)",
        legend=dict(
            x=0.01,
            y=0.99,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="rgba(0,0,0,0.1)",
            borderwidth=1,
        ),
        template="plotly_white",
        # Extra right side padding (N + 70) to let the labels sit cleanly off the data lines
        xaxis=dict(range=[-10, max_N+14]),
        shapes=day_lines,
    )

    fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(230, 230, 230, 0.4)",
        dtick=24,
    )

    fig.update_yaxes(title_text="Profit Cumulatif (€)")

    fig.show()
    fig.write_image("2_2_cumulative.pdf", format="pdf", width=800, height=500)

In [89]:
# results_dict = {
#     0: hydro_with_cost("BelgiumScenario1_15jours.txt", C_jour=0),
#     50000: hydro_with_cost("BelgiumScenario1_15jours.txt", C_jour=50000),
#     150000: hydro_with_cost("BelgiumScenario1_15jours.txt", C_jour=150000),
# }


plot_hydro_profits(results_dict, "BelgiumScenario1_15jours.txt")

In [82]:
result = hydro_with_cost("BelgiumScenario1_15jours.txt",150000)
print(result["C"])
print(result["M"][119:145])




[1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 1. 0. 0. 1.]
[     0.      0.      0.      0.      0.      0.      0.      0.      0.
      0. 500000. 500000. 500000. 500000. 500000. 500000. 500000. 500000.
      0.      0.      0.      0.      0.      0.      0.      0.]


In [62]:
result = hydro_with_cost("BelgiumScenario1_15jours.txt",50000)
plot_hydro_profits(result, "BelgiumScenario1_15jours.txt", 50000)
print(result["C"])
print(result["M"][119:145])

[1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[     0.              0.              0.              0.
      0.              0.              0.              0.
      0.              0.         500000.         500000.
 500000.         500000.         473454.43381933 500000.
 500000.         500000.              0.              0.
      0.              0.              0.              0.
      0.              0.        ]


In [63]:
result = hydro_with_cost("BelgiumScenario1_15jours.txt",150000)
plot_hydro_profits(result, "BelgiumScenario1_15jours.txt", 150000)
print(result["C"])
print(result["M"][119:145])

[1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 1. 0. 0. 1.]
[     0.      0.      0.      0.      0.      0.      0.      0.      0.
      0. 500000. 500000. 500000. 500000. 500000. 500000. 500000. 500000.
      0.      0.      0.      0.      0.      0.      0.      0.]


In [ ]:
def hydro_baseline(data):#flux turbiné compense exactement le flux entrant à chaque instant.
    params = read_data(data)
    return sum([params["P"][t]*params["et"]*params["F"][t] for t in range(params["n"])])

np.float64(8119553.744095378)

In [ ]:
hydro_baseline("BelgiumScenario1.txt") #Pareil pour both scenarios

np.float64(8119553.744095378)

In [98]:
def hydro_sans_pompage(data): #Same as hydro mais sans pompage whatsoever
    params = read_data(data)
    N = int(params['n']) 
    
    #Decision variables
    T = cp.Variable(N, nonneg=True, name="Turbinage")
    D = cp.Variable(N, nonneg=True, name="Délestage")
    V = cp.Variable(N, nonneg=True, name="Volume")
    
    #Objective function
    profits = params['P'] @ (params['et'] * T)
    objective = cp.Maximize(profits)
    
    #list of constraints
    constraints = []
    
    #Cyclic volume constraint
    constraints += [V[0] == params['v0']]
    constraints += [V[N-1] == params['v0']]
    
    for t in range(N):
        
        #Limits 
        constraints += [V[t] >= params['vmin'], V[t] <= params['vmax']]
        constraints += [T[t] <= params['tmax']]
        constraints += [D[t] <= params['dmax']]
        
        #TDlimit
        constraints += [T[t] + D[t] >= params['tdmin']]

        if t > 0:
            #volume reservoir (V0 quand t == 0)
            constraints += [V[t] == V[t-1] + params['F'][t] - T[t] - D[t]]
            #Derivative limits
            constraints += [(T[t] - T[t-1]) <= params['vtmax']]
            constraints += [(T[t] - T[t-1]) >= -params['vtmax']]
            constraints += [(D[t] - D[t-1]) <= params['vdmax']]
            constraints += [(D[t] - D[t-1]) >= -params['vdmax']]
            
    #solver
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)
    
    sol = {
        "V": V.value,       
        "T": T.value,       
        "D": D.value,              
        "valopt": prob.value 
    }
    
    return sol

In [103]:
hydro_sans_pompage("BelgiumScenario1.txt")

{'V': array([5000000.        , 4893975.77028029, 4937951.54056058, ...,
        4920000.        , 4960000.        , 5000000.        ]),
 'T': array([400000., 200000.,  50000., ..., 250000.,  50000.,  50000.]),
 'D': array([0., 0., 0., ..., 0., 0., 0.]),
 'valopt': np.float64(9601306.915573996)}

In [104]:
hydro_sans_pompage("BelgiumScenario2.txt")

{'V': array([5000000.        , 4743975.77028029, 4537951.54056058, ...,
        5461033.80189058, 5205516.90094529, 5000000.        ]),
 'T': array([400000.        , 350000.        , 300000.        , ...,
        295516.90094529, 345516.90094529, 295516.90094529]),
 'D': array([0., 0., 0., ..., 0., 0., 0.]),
 'valopt': np.float64(9464043.273290971)}

In [129]:
def hydro_with_duals(data_file):
    params = read_data(data_file)
    N = int(params['n']) 
    
    T = cp.Variable(N, nonneg=True)
    M = cp.Variable(N, nonneg=True)
    D = cp.Variable(N, nonneg=True)
    V = cp.Variable(N, nonneg=True)
    
    profits = params['P'] @ (params['et'] * T - params['me'] * M)
    objective = cp.Maximize(profits)
    
    constraints = []
    
    #put constraintes in variables to evaluate later
    c_vmax = [V <= params['vmax']]
    c_tmax = [T <= params['tmax']]
    c_mmax = [M <= params['mmax']]
    constraints += c_vmax + c_tmax + c_mmax
    
    #Water Volume
    c_volume = [V[0] == params['v0']]
    c_vtmax = []
    c_vdmax = []
    for t in range(1, N):
        c_volume.append(V[t] == V[t-1] + params['F'][t] + M[t] - T[t] - D[t])
        c_vtmax.append(T[t] - T[t-1] <= params['vtmax'])
        c_vtmax.append(T[t] - T[t-1] >= -params['vtmax'])
        c_vdmax.append(D[t] - D[t-1] <= params['vdmax'])
        c_vdmax.append(D[t] - D[t-1] >= -params['vdmax'])
    
    constraints += c_volume
    constraints += c_vtmax
    constraints += c_vdmax
    constraints += [V[N-1] == params['v0']]
    constraints += [V >= params['vmin']]
    constraints += [D <= params['dmax']]
    constraints += [T + D >= params['tdmin']]
    
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)
    

    #Valeurs duals
    #a)
    val_vmax = sum(c_vmax[0].dual_value)
    val_tmax = sum(c_tmax[0].dual_value)
    val_mmax = sum(c_mmax[0].dual_value)
    val_vtmax = sum([abs(c.dual_value) for c in c_vtmax])
    val_fk = sum([c.dual_value for c in c_volume])
    val_Pk = sum((params['et'] * T.value - params['me'] * M.value))
    
    print("Impact Vmax:",val_vmax,"€/m3")
    print(val_vmax/params['vmax'])
    print("Impact Tmax:",val_tmax,"€/(m3/h)")
    print(val_tmax/params['tmax'])
    print("Impact Mmax:",val_mmax,"€/(m3/h)")
    print(val_mmax/params['mmax'])
    print("Impact VTmax",val_vtmax,"€/(m3/h2)")
    print(val_vtmax/params['vtmax'])
    print("Impact Fk:",val_fk,"€/(m3/h)")
    print(val_fk/sum(params['F']))
    print("Impact Pk:",val_Pk,"€")
    print(val_Pk/prob.value)


In [130]:
hydro_with_duals("BelgiumScenario1.txt")

Impact Vmax: 0.5650205000000003 €/m3
9.417008333333338e-08
Impact Tmax: 3.0536385000000013 €/(m3/h)
7.634096250000003e-06
Impact Mmax: 0.9906275000000003 €/(m3/h)
1.9812550000000007e-06
Impact VTmax 0.6905314999999999 €/(m3/h2)
3.452657499999999e-06
Impact Fk: 82.689577 €/(m3/h)
5.541743932410368e-07
Impact Pk: 23931.093339291045 €
0.002156711880079816


In [131]:
hydro_with_duals("BelgiumScenario2.txt")

Impact Vmax: 0.19626375801315654 €/m3
2.453296975164457e-08
Impact Tmax: 0.9516609711373228 €/(m3/h)
2.379152427843307e-06
Impact Mmax: 2.027683346130895 €/(m3/h)
4.05536669226179e-06
Impact VTmax 12.97170236000215 €/(m3/h2)
0.000259434047200043
Impact Fk: 83.53558791452438 €/(m3/h)
5.598442442941126e-07
Impact Pk: 23540.448231890616 €
0.002060359673332122


In [133]:
hydro("extremeScenario.txt") #quitupler

{'V': array([5000000.        , 4243975.77028029, 4000000.        , ...,
        5120000.        , 4960000.        , 5000000.        ]),
 'T': array([1050000.,  850000.,  650000., ...,  450000.,  250000.,   50000.]),
 'D': array([0., 0., 0., ..., 0., 0., 0.]),
 'M': array([     0.        ,      0.        , 312048.45943942, ...,
             0.        ,      0.        ,      0.        ]),
 'valopt': np.float64(11482623.21760554)}